## Spending where the outcome is in doubt

TPE again, with repetitions allocated by how much nearby cells disagreed rather than
uniformly. Read it against `nav_search_tpe`: same strategy, same space, same *run* budget.

**What to look for:** an uneven bar chart. Cells whose neighbours agreed get one run; cells
near the boundary get several. On the campaign that motivated this, 3 of 32 cells produced
a mixed outcome over 5 repetitions — the other 145 runs each bought a bit that one run had
already established.


In [ ]:
# DATA_DIR is replaced by the service with the node being viewed. The assignment must stay
# a plain literal for that substitution to work.
DATA_DIR = ''

import json
import pandas as pd
import matplotlib.pyplot as plt

from robovast.common.analysis import CampaignDataError, open_campaign_store

def load_units(data_dir):
    """One row per evaluated cell: its parameters, objectives and measures.

    Read from the campaign's own store rather than the results index because that is where a
    SEARCH records what it scored -- the index holds per-run tables, and a search's unit of
    analysis is the cell. The store is also written as the search runs, so this works on a
    campaign that is still going or was never postprocessed.
    """
    # open_campaign_store rather than a sqlite3.connect on a path built here: it resolves the
    # campaign ROOT from data_dir, so this cell also works at a configuration node instead of
    # only at the campaign, and it is the one place that knows where the store lives.
    try:
        conn = open_campaign_store(data_dir)
    except CampaignDataError as exc:
        # Reported, not swallowed. "This campaign scored nothing" and "its record is not
        # here" are different answers and only the first is a result -- an empty frame
        # returned quietly reads as the first while meaning the second.
        print(f'[no data] {exc}')
        return pd.DataFrame()
    try:
        units = pd.read_sql_query(
            "SELECT u.paramset_id, u.config_name, u.params_json, u.objectives_json,"
            "       u.measures_json, u.n_samples, u.status, b.idx AS batch"
            "  FROM unit u LEFT JOIN batch b ON b.id = u.batch_id"
            " ORDER BY b.idx, u.id", conn)
    finally:
        conn.close()
    if units.empty:
        return units
    for col, prefix in (('params_json', ''), ('objectives_json', ''), ('measures_json', 'm_')):
        expanded = units[col].apply(lambda s: json.loads(s) if s else {}).apply(pd.Series)
        expanded.columns = [f'{prefix}{c}' for c in expanded.columns]
        units = pd.concat([units.drop(columns=[col]), expanded], axis=1)
    return units

units = load_units(DATA_DIR)
scored = units[units['status'] == 'evaluated'] if 'status' in units else units
print(f"{len(units)} cell(s) recorded, {len(scored)} scored")

if scored.empty:
    print("No scored cells yet. A search records a cell once its batch has been evaluated;"
          "\nif this campaign failed early, its controller log says why.")


In [ ]:
TITLE = 'Adaptive repetitions — the same answer for less'

# How many repetitions each cell was given. A flat line is a campaign that spent the same
# everywhere; a spread one is a campaign that spent where the outcome was in doubt.
if not scored.empty:
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.bar(range(len(scored)), scored['n_samples'], color='tab:blue', alpha=0.8)
    ax.set_xlabel('cell (in evaluation order)'); ax.set_ylabel('repetitions run')
    ax.set_title('%s: %d runs over %d cells'
                 % (TITLE, int(scored['n_samples'].sum()), len(scored)))
    plt.tight_layout(); plt.show()


In [ ]:
if not scored.empty:
    failed = (scored['robustness'] < 0).sum()
    print(f"cells scored          : {len(scored)}")
    print(f"runs spent            : {int(scored['n_samples'].sum())}")
    print(f"cells that failed     : {failed}  ({failed / len(scored):.0%})")
    print(f"worst robustness      : {scored['robustness'].min():.3f}")
    print()
    reps = scored['n_samples']
    spent = int(reps.sum())
    flat = 3 * len(scored)
    print("What this campaign is FOR -- the same answer for FEWER runs:")
    print(f"  repetitions per cell : mean {reps.mean():.2f}, range {reps.min()}..{reps.max()}")
    print(f"  runs spent           : {spent}")
    print(f"  a flat 3 reps/cell   : {flat}   ({spent - flat:+d} runs)")
    print("  The saving is real only if the worst crossing is as deep as the campaign this")
    print("  is read against found with MORE runs. Compare both numbers, never one.")
